# FedProx poisoning robustness (BoT-IoT): random noise injection

## 1. Imports

In [1]:
import os
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

import flwr as fl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

warnings.filterwarnings("ignore")

## 2. Configuration

In [2]:
CSV_PATH = r"../../../data/Bot-IoT.csv"
TARGET_MULTICLASS = "category"
NORMAL_CLASS = "Normal"
DROP_COLS = ['attack', 'category', 'subcategory ', 'pkSeqID', 'saddr', 'daddr', 'soui', 'doui', 'sco', 'dco', 'smac', 'dmac']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_CLIENTS = 10
NUM_PARTITIONS = 10
BATCH_SIZE = 32
EPOCHS = 5
EPSILON = 1e-8
LEARNING_RATE = 0.001


BINARY = False
IID = False
DIRICHLET_ALPHA = 0.5

BASE_SEED = 2024
NUM_ROUNDS = 15

GPU_PER_CLIENT = 0.5 if torch.cuda.is_available() else 0.0
CPUS_PER_CLIENT = max(1, (os.cpu_count() or 2) // 2)

ATTACK = "none"          # "none" | "label_flip" | "noise"
MALICIOUS_FRAC = 0.0
FLIP_PROB = 0.0
FLIP_MODE = "random"
SOURCE_CLASS = 0
TARGET_CLASS = 1
POISON_SEED = 2024
MALICIOUS_CLIENTS = set()

# Random noise injection (features only; labels untouched)
NOISE_MODE = "gaussian"   # "gaussian" | "uniform" (uniform is scaled to the same std)
NOISE_STD = 0.0           # in units of per-feature std (features are already standardized)
NOISE_PROB = 1.0          # fraction of a malicious client's samples that receive noise
NOISE_STATIC = True       # True: fixed poisoned dataset; False: fresh noise on every access

FEDPROX_MU = 0.01


Using device: cuda


## 3. Data loading

In [3]:
def load_dataset(file_path, target_multiclass, normal_class, binary,
                 drop_cols, test_size=0.3, random_state=42):
    df = pd.read_csv(file_path)
    df = df.drop_duplicates()

    df = df.dropna(subset=[target_multiclass])
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    for col in categorical_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode()
            df[col] = df[col].fillna(mode_val[0] if not mode_val.empty else "Unknown")

    y_multi = df[target_multiclass].astype(str).str.strip()
    if binary:
        y = np.where(y_multi.str.lower() == normal_class.lower(), "Benign", "Attack")
        y = pd.Series(y, index=df.index)
    else:
        y = y_multi

    X = df.drop(columns=drop_cols, errors="ignore").copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y)

    non_numeric_cols = list(
        set(X_train.select_dtypes(exclude=[np.number]).columns.tolist())
        | set(X_test.select_dtypes(exclude=[np.number]).columns.tolist()))
    feature_encoders = {}
    for col in non_numeric_cols:
        le_col = LabelEncoder()
        le_col.fit(X_train[col].astype(str))
        feature_encoders[col] = le_col
        mapping = {cls: idx for idx, cls in enumerate(le_col.classes_)}
        X_train[col] = le_col.transform(X_train[col].astype(str))
        X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1).astype(int)

    def safe_numeric(df_):
        df_ = df_.apply(lambda c: c.map(lambda v: str(v).strip() if isinstance(v, str) else v))
        df_ = df_.apply(pd.to_numeric, errors="coerce")
        return df_.replace([np.inf, -np.inf], np.nan).fillna(0)

    X_train = safe_numeric(X_train)
    X_test = safe_numeric(X_test)

    global INPUT_DIM
    INPUT_DIM = X_train.shape[1]

    y_train = pd.Series(np.asarray(y_train)).astype(str).str.strip()
    y_test = pd.Series(np.asarray(y_test)).astype(str).str.strip()
    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train.values)
    y_test_enc = label_encoder.transform(y_test.values)
    class_names = label_encoder.classes_
    num_classes = len(class_names)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train.values.astype(np.float64))
    X_test_scaled = scaler.transform(X_test.values.astype(np.float64))

    train_dataset = TensorDataset(torch.from_numpy(X_train_scaled).float(),
                                  torch.from_numpy(y_train_enc).long())
    test_dataset = TensorDataset(torch.from_numpy(X_test_scaled).float(),
                                 torch.from_numpy(y_test_enc).long())
    print(f"Classes ({num_classes}): {list(class_names)}")
    print(f"Features: {INPUT_DIM} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")
    return (train_dataset, test_dataset, class_names, num_classes,
            scaler, label_encoder, feature_encoders)


(
    train_dataset, test_dataset, class_names, NUM_CLASSES,
    scaler, label_encoder, feature_encoders,
) = load_dataset(CSV_PATH, TARGET_MULTICLASS, NORMAL_CLASS, BINARY, DROP_COLS)

Classes (4): ['DDoS/DoS', 'Normal', 'Reconnaissance', 'Theft']
Features: 23 | Train: 35791 | Test: 15339


## 4. Partitioning (IID and Non-IID)

In [4]:
def partition_dataset_iid(dataset, num_partitions):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    indices_by_class = [[] for _ in range(NUM_CLASSES)]
    for idx, label in enumerate(labels):
        indices_by_class[label].append(idx)
    partitions = [[] for _ in range(num_partitions)]
    for c in range(NUM_CLASSES):
        indices = indices_by_class[c]
        np.random.shuffle(indices)
        per = len(indices) // num_partitions
        rem = len(indices) % num_partitions
        start = 0
        for p in range(num_partitions):
            extra = 1 if p < rem else 0
            end = start + per + extra
            partitions[p].extend(indices[start:end])
            start = end
    for p in range(num_partitions):
        np.random.shuffle(partitions[p])
    return partitions


def partition_dataset_dirichlet(dataset, num_partitions, dirichlet_alpha):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    indices_by_class = [[] for _ in range(NUM_CLASSES)]
    for idx, label in enumerate(labels):
        indices_by_class[label].append(idx)
    partitions = [[] for _ in range(num_partitions)]
    for c in range(NUM_CLASSES):
        indices = indices_by_class[c]
        np.random.shuffle(indices)
        proportions = np.random.dirichlet([dirichlet_alpha] * num_partitions)
        counts = (proportions * len(indices)).astype(int)
        diff = len(indices) - counts.sum()
        if diff > 0:
            for k in np.argsort(proportions)[-diff:]:
                counts[k] += 1
        elif diff < 0:
            for k in np.argsort(proportions)[:abs(diff)]:
                if counts[k] > 0:
                    counts[k] -= 1
        start = 0
        for p in range(num_partitions):
            end = start + counts[p]
            partitions[p].extend(indices[start:end])
            start = end
    for p in range(num_partitions):
        np.random.shuffle(partitions[p])
    return partitions


def partition_dataset(dataset, num_partitions):
    if IID:
        return partition_dataset_iid(dataset, num_partitions)
    return partition_dataset_dirichlet(dataset, num_partitions, DIRICHLET_ALPHA)


train_partitions = partition_dataset(train_dataset, NUM_PARTITIONS)
print(f"Created {len(train_partitions)} partitions ({'IID' if IID else 'Non-IID'})")

Created 10 partitions (Non-IID)


## 5. Model, parameters, and evaluation

In [5]:
class model(nn.Module):
    def __init__(self, INPUT_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, 50)
        self.fc2 = nn.Linear(50, 25)
        self.fc3 = nn.Linear(25, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


def get_ndarrays(net):
    return [val.detach().cpu().numpy() for _, val in net.state_dict().items()]


def set_ndarrays(net, params):
    state_dict = net.state_dict()
    new_state_dict = {k: torch.tensor(v, device=device)
                      for k, v in zip(state_dict.keys(), params)}
    net.load_state_dict(new_state_dict, strict=True)


@torch.no_grad()
def evaluate_global_model(params, test_loader):
    net = model(INPUT_DIM, NUM_CLASSES).to(device)
    set_ndarrays(net, fl.common.parameters_to_ndarrays(params)
                 if not isinstance(params, list) else params)
    net.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, total = 0.0, 0
    y_true, y_pred = [], []
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = net(xb)
        total_loss += loss_fn(logits, yb).item() * yb.size(0)
        total += yb.size(0)
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(logits.argmax(dim=1).cpu().numpy())
    return total_loss / max(1, total), {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

## 6. Label-flipping wrapper

In [6]:
class LabelFlippedDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, num_classes, flip_prob=1.0, mode="random",
                 source_class=0, target_class=1, seed=0):
        self.base = base_dataset
        self.num_classes = int(num_classes)
        self.flip_prob = float(flip_prob)
        self.mode = str(mode)
        self.source_class = int(source_class)
        self.target_class = int(target_class)
        self.rng = np.random.RandomState(seed)

    def __len__(self):
        return len(self.base)

    def _flip_label(self, y):
        if self.mode == "targeted":
            return self.target_class if y == self.source_class else y
        new_y = y
        while new_y == y:
            new_y = int(self.rng.randint(0, self.num_classes))
        return new_y

    def __getitem__(self, idx):
        x, y = self.base[idx]
        y_int = int(y.item()) if torch.is_tensor(y) else int(y)
        if self.rng.rand() < self.flip_prob:
            y_int = self._flip_label(y_int)
        return x, torch.tensor(y_int, dtype=torch.long)

### 6b. Random noise injection wrapper and attack dispatcher

In [7]:
class NoiseInjectedDataset(torch.utils.data.Dataset):
    """Random-noise-injection data poisoning: perturbs the FEATURES of a malicious
    client's samples, labels are left untouched.

    Features are already standardized, so noise_std is in units of per-feature std.
    static=True  -> noise is drawn once at construction, i.e. the attacker owns a fixed
                    poisoned dataset (same noise every epoch and every round).
    static=False -> fresh noise on every access; this behaves like noise augmentation
                    and is a much weaker attack.
    """

    def __init__(self, base_dataset, noise_std=1.0, noise_prob=1.0, mode="gaussian",
                 static=True, seed=0):
        self.base = base_dataset
        self.noise_std = float(noise_std)
        self.noise_prob = float(noise_prob)
        self.mode = str(mode)
        self.static = bool(static)
        self.rng = np.random.RandomState(seed)
        self.noise = None
        if self.static and len(base_dataset) > 0:
            n = len(base_dataset)
            feat_shape = tuple(base_dataset[0][0].shape)
            noise = self._draw((n,) + feat_shape)
            hit = (self.rng.rand(n) < self.noise_prob).astype(np.float32)
            hit = hit.reshape((n,) + (1,) * len(feat_shape))
            self.noise = torch.from_numpy((noise * hit).astype(np.float32))

    def _draw(self, shape):
        if self.mode == "uniform":
            a = self.noise_std * math.sqrt(3.0)          # U(-a, a) has std noise_std
            return self.rng.uniform(-a, a, size=shape)
        return self.rng.randn(*shape) * self.noise_std   # gaussian

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]
        if self.static:
            x = x + self.noise[idx]
        elif self.rng.rand() < self.noise_prob:
            x = x + torch.from_numpy(self._draw(tuple(x.shape)).astype(np.float32))
        return x, y


def build_client_dataset(base_subset, cid_int):
    """Returns (dataset, is_malicious) for one client. New attacks plug in here."""
    is_malicious = (ATTACK != "none") and (cid_int in MALICIOUS_CLIENTS)
    if not is_malicious:
        return base_subset, False
    if ATTACK == "label_flip":
        return LabelFlippedDataset(
            base_dataset=base_subset, num_classes=NUM_CLASSES,
            flip_prob=FLIP_PROB, mode=FLIP_MODE,
            source_class=SOURCE_CLASS, target_class=TARGET_CLASS,
            seed=POISON_SEED + cid_int), True
    if ATTACK == "noise":
        return NoiseInjectedDataset(
            base_subset, noise_std=NOISE_STD, noise_prob=NOISE_PROB,
            mode=NOISE_MODE, static=NOISE_STATIC,
            seed=POISON_SEED + cid_int), True
    raise ValueError(f"Unknown ATTACK: {ATTACK}")


## 7. Flower client

In [8]:
def client_fn(cid):
    cid_int = int(cid)
    partition_indices = train_partitions[cid_int]
    base_subset = Subset(train_dataset, partition_indices)

    client_dataset, is_malicious = build_client_dataset(base_subset, cid_int)

    train_loader = DataLoader(client_dataset, batch_size=BATCH_SIZE, shuffle=True)

    class BaselineClient(fl.client.NumPyClient):
        def __init__(self):
            self.net = model(INPUT_DIM, NUM_CLASSES).to(device)
            self.train_loader = train_loader
            self.is_malicious = is_malicious

        def get_parameters(self, config=None):
            return get_ndarrays(self.net)

        def fit(self, parameters, config):
            set_ndarrays(self.net, parameters)
            global_weights = [torch.tensor(p, device=device) for p in parameters]
            self.net.train()
            opt = optim.Adam(self.net.parameters(), lr=LEARNING_RATE,weight_decay=1e-4)
            loss_fn = nn.CrossEntropyLoss()
            total_loss, total_seen = 0.0, 0
            for _ in range(EPOCHS):
                for xb, yb in self.train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    opt.zero_grad()
                    loss = loss_fn(self.net(xb), yb)
                    prox = 0.0
                    for w, w_g in zip(self.net.parameters(), global_weights):
                        prox = prox + ((w - w_g) ** 2).sum()
                    loss = loss + (FEDPROX_MU / 2.0) * prox
                    loss.backward()
                    opt.step()
                    total_loss += loss.item() * yb.size(0)
                    total_seen += yb.size(0)
            avg_train_loss = total_loss / max(1, total_seen)
            return (get_ndarrays(self.net), len(client_dataset),
                    {"train_loss": float(avg_train_loss),
                      "is_malicious": int(self.is_malicious)})

        def evaluate(self, parameters, config):
            return 0.0, len(client_dataset), {}

    return BaselineClient().to_client()

## 8. Evaluation history

In [9]:
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

eval_rounds, eval_loss, eval_acc, eval_prec, eval_rec, eval_f1 = [], [], [], [], [], []


def reset_histories():
    global eval_rounds, eval_loss, eval_acc, eval_prec, eval_rec, eval_f1
    eval_rounds, eval_loss, eval_acc, eval_prec, eval_rec, eval_f1 = [], [], [], [], [], []

## 9. Strategy

In [10]:
def make_strategy():
    def evaluate_fn(server_round, parameters, config):
        loss, metrics = evaluate_global_model(parameters, test_loader)
        eval_rounds.append(server_round)
        eval_loss.append(loss)
        eval_acc.append(metrics["accuracy"])
        eval_prec.append(metrics["precision"])
        eval_rec.append(metrics["recall"])
        eval_f1.append(metrics["f1"])
        print(f"[FedProx][Round {server_round}] loss={loss:.4f} "
              f"acc={metrics['accuracy']:.4f} f1={metrics['f1']:.4f}")
        return loss, metrics

    return fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        evaluate_fn=evaluate_fn,
    )

## 10. Poisoning helper

In [11]:
def set_poisoning(mal_frac, attack="noise", seed=2024, *,
                  noise_std=1.0, noise_prob=1.0, noise_mode="gaussian", noise_static=True,
                  flip_prob=1.0, flip_mode="random", source_class=0, target_class=1):
    """Choose the malicious clients and configure the attack they run.
    attack: "none" | "noise" | "label_flip"."""
    global ATTACK, MALICIOUS_FRAC, MALICIOUS_CLIENTS, POISON_SEED
    global NOISE_STD, NOISE_PROB, NOISE_MODE, NOISE_STATIC
    global FLIP_PROB, FLIP_MODE, SOURCE_CLASS, TARGET_CLASS
    POISON_SEED = int(seed)
    MALICIOUS_FRAC = float(mal_frac)
    NOISE_STD, NOISE_PROB = float(noise_std), float(noise_prob)
    NOISE_MODE, NOISE_STATIC = str(noise_mode), bool(noise_static)
    FLIP_PROB, FLIP_MODE = float(flip_prob), str(flip_mode)
    SOURCE_CLASS, TARGET_CLASS = int(source_class), int(target_class)

    num_mal = int(NUM_CLIENTS * MALICIOUS_FRAC)
    inert = ((attack == "noise" and (NOISE_STD <= 0 or NOISE_PROB <= 0))
             or (attack == "label_flip" and FLIP_PROB <= 0))
    ATTACK = "none" if (num_mal <= 0 or inert) else str(attack)

    rng = np.random.RandomState(POISON_SEED)
    if ATTACK == "none":
        MALICIOUS_CLIENTS = set()
    else:
        MALICIOUS_CLIENTS = set(rng.choice(np.arange(NUM_CLIENTS),
                                           size=num_mal, replace=False).tolist())
    if ATTACK == "noise":
        detail = (f"noise_mode={NOISE_MODE}, noise_std={NOISE_STD}, "
                  f"noise_prob={NOISE_PROB}, static={NOISE_STATIC}")
    elif ATTACK == "label_flip":
        detail = f"flip_prob={FLIP_PROB}, flip_mode={FLIP_MODE}"
    else:
        detail = "no attackers"
    print(f"[Poison] attack={ATTACK}, mal_frac={MALICIOUS_FRAC}, {detail}, "
          f"malicious_clients={sorted(MALICIOUS_CLIENTS)}")


## 11. Experiment runner

In [12]:
def run_one_experiment(num_rounds=15, seed=2024):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    reset_histories()
    strategy = make_strategy()
    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=num_rounds),
        strategy=strategy,
        client_resources={"num_cpus": CPUS_PER_CLIENT, "num_gpus": GPU_PER_CLIENT},
    )
    if len(eval_rounds) == 0:
        return None
    return {
        "final_round": int(eval_rounds[-1]),
        "final_loss": float(eval_loss[-1]),
        "final_accuracy": float(eval_acc[-1]),
        "final_precision": float(eval_prec[-1]),
        "final_recall": float(eval_rec[-1]),
        "final_f1": float(eval_f1[-1]),
        "rounds": list(eval_rounds),
        "acc_curve": list(eval_acc),
        "loss_curve": list(eval_loss),
    }

## 12. Rounds and seed

In [13]:
NUM_ROUNDS = 15
BASE_SEED = 2024

## 13. Poisoning sweep (random noise injection)

In [ ]:
mal_fracs   = [0.2, 0.4, 0.6, 0.8]
noise_stds  = [5.0]   # in units of per-feature std (features are standardized)
noise_probs = [1.0]        # fraction of a malicious client's samples that get noise
noise_mode  = "gaussian"   # or "uniform" (same std)

# (mal_frac, noise_std, noise_prob); the first entry is the clean reference (no attackers)
configs = [(mf, s, p) for mf in mal_fracs
                               for s in noise_stds for p in noise_probs]

results = []
curves = {}
for mf, std, p in configs:
    set_poisoning(mal_frac=mf, attack="noise", seed=BASE_SEED,
                  noise_std=std, noise_prob=p, noise_mode=noise_mode)
    res = run_one_experiment(num_rounds=NUM_ROUNDS, seed=BASE_SEED)
    if res is None:
        continue
    results.append({
        "algo": "FedProx",
        "attack": "noise" if mf > 0 else "none",
        "noise_mode": noise_mode,
        "mal_frac": mf,
        "noise_std": std,
        "noise_prob": p,
        "final_accuracy": res["final_accuracy"],
        "final_f1": res["final_f1"],
        "final_precision": res["final_precision"],
        "final_recall": res["final_recall"],
        "final_loss": res["final_loss"],
    })
    curves[(mf, std, p)] = (res["rounds"], res["acc_curve"])
    print(f"[FedProx Sweep] mal_frac={mf:.2f} std={std} acc={res['final_accuracy']:.4f}")

df_results = pd.DataFrame(results).sort_values(
    ["mal_frac", "noise_std", "noise_prob"]).reset_index(drop=True)
df_results.to_csv("fedprox_botiot_noise.csv", index=False)
df_results


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=15, no round_timeout


[Poison] attack=noise, mal_frac=0.2, noise_mode=gaussian, noise_std=5.0, noise_prob=1.0, static=True, malicious_clients=[2, 7]


2026-09-21 14:19:36,229	INFO worker.py:1771 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'node:172.24.90.50': 1.0, 'accelerator_type:G': 1.0, 'node:__internal_head__': 1.0, 'CPU': 20.0, 'object_store_memory': 2956024627.0, 'memory': 5912049255.0, 'GPU': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 10, 'num_gpus': 0.5}
INFO :      Flower VCE: Creating VirtualClientEngineActorPool with 2 actors
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
(ClientAppActor pid=173028) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=173028) 
(ClientAppActor pid=173028)

[FedProx][Round 0] loss=1.4457 acc=0.0520 f1=0.0477


(ClientAppActor pid=173028) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=173028) 
(ClientAppActor pid=173028)             This is a deprecated feature. It will be removed
(ClientAppActor pid=173028)             entirely in future versions of Flower.
(ClientAppActor pid=173028)         
(ClientAppActor pid=173029) 
(ClientAppActor pid=173029)         
(ClientAppActor pid=173029) 
(ClientAppActor pid=173029)         
(ClientAppActor pid=173028) 
(ClientAppActor pid=173028)         
(ClientAppActor pid=173029) 
(ClientAppActor pid=173029)         
(ClientAppActor pid=173029) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` lik

...

[FedProx][Round 14] loss=0.7670 acc=0.8205 f1=0.5375


(ClientAppActor pid=179524) 
(ClientAppActor pid=179524)         
(ClientAppActor pid=179524) 
(ClientAppActor pid=179524)         
(ClientAppActor pid=179524) 
(ClientAppActor pid=179524)         
(ClientAppActor pid=179525) 
(ClientAppActor pid=179525)         
(ClientAppActor pid=179525) 
(ClientAppActor pid=179525)         
(ClientAppActor pid=179525) 
(ClientAppActor pid=179525)         
(ClientAppActor pid=179525) 
(ClientAppActor pid=179525)         
(ClientAppActor pid=179525) 
(ClientAppActor pid=179525)         
(ClientAppActor pid=179524) 
(ClientAppActor pid=179524)         
(ClientAppActor pid=179524) 
(ClientAppActor pid=179524)         
(ClientAppActor pid=179524) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 17x across cluster]
(ClientAppActor pid=179524)           

[FedProx][Round 15] loss=0.7654 acc=0.8196 f1=0.5367
[FedProx Sweep] mal_frac=0.80 std=5.0 acc=0.8196


,algo,attack,noise_mode,mal_frac,noise_std,noise_prob,final_accuracy,final_f1,final_precision,final_recall,final_loss
0,FedProx,noise,gaussian,0.2,5.0,1.0,0.986961,0.984546,0.989328,0.980101,0.061264
1,FedProx,noise,gaussian,0.4,5.0,1.0,0.911598,0.727632,0.941951,0.698802,0.185696
2,FedProx,noise,gaussian,0.6,5.0,1.0,0.879588,0.635880,0.675866,0.631516,0.405864
3,FedProx,noise,gaussian,0.8,5.0,1.0,0.819610,0.536695,0.650865,0.552002,0.765373


(raylet) [2026-09-21 14:37:31,720 E 178411 178411] (raylet) node_manager.cc:3064: 1 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 907d108c78aa15a7973862f384eed67bc60977766752e3a676e426ad, IP: 172.24.90.50) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 172.24.90.50`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.
